# Day 4: Performance Analytics
## Mutual Fund Risk & Return Analysis

This notebook computes performance metrics for all 40 mutual fund schemes using NAV history and benchmark index data (2022–2025).

**Metrics covered:**
1. Daily returns
2. CAGR (1-year, 3-year, 5-year)
3. Sharpe Ratio
4. Sortino Ratio
5. Alpha & Beta vs Nifty 100
6. Maximum Drawdown
7. Fund Scorecard (0–100)
8. Benchmark comparison chart (top 5 funds vs Nifty 50 & Nifty 100)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

RISK_FREE_RATE = 0.065
TRADING_DAYS = 252
DATA_DIR = Path("data/processed")
CHART_DIR = Path("reports/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
nav_df = pd.read_csv(DATA_DIR / "02_nav_history.csv", parse_dates=["date"])
fund_df = pd.read_csv(DATA_DIR / "01_fund_master.csv")
bench_df = pd.read_csv(DATA_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

print(f"NAV records: {len(nav_df):,}")
print(f"Funds: {nav_df['amfi_code'].nunique()}")
print(f"Date range: {nav_df['date'].min().date()} to {nav_df['date'].max().date()}")
print(f"Benchmark indices: {bench_df['index_name'].unique().tolist()}")

## 1. Daily Returns

**Daily return** measures how much a fund's NAV changed from one trading day to the next:

$$\text{daily\_return} = \frac{\text{NAV}_t}{\text{NAV}_{t-1}} - 1$$

We compute this for every fund using `nav.shift(1)` to get the previous day's NAV. The first observation for each fund will be missing (NaN) because there is no prior NAV.

We validate the distribution using summary statistics (mean, std, min, max) and a histogram to check for outliers or unusual patterns.

In [ ]:
nav_df = nav_df.sort_values(["amfi_code", "date"])
nav_df["daily_return"] = nav_df.groupby("amfi_code")["nav"].transform(
    lambda x: (x / x.shift(1)) - 1
)

returns_summary = nav_df.groupby("amfi_code")["daily_return"].agg(
    ["count", "mean", "std", "min", "max"]
).reset_index()
returns_summary = returns_summary.merge(
    fund_df[["amfi_code", "scheme_name"]], on="amfi_code"
)
print("Daily return summary statistics (per fund):")
display(returns_summary.head(10))
print("\nOverall pooled daily returns:")
print(nav_df["daily_return"].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
all_returns = nav_df["daily_return"].dropna()
ax.hist(all_returns, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(all_returns.mean(), color="red", linestyle="--", label=f"Mean: {all_returns.mean():.4f}")
ax.set_xlabel("Daily Return")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Daily Returns (All 40 Funds)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. CAGR (Compound Annual Growth Rate)

**CAGR** tells us the average yearly growth rate of a fund, assuming profits are reinvested:

$$\text{CAGR} = \left(\frac{\text{Ending NAV}}{\text{Starting NAV}}\right)^{\frac{1}{\text{years}}} - 1$$

We compute CAGR for **1-year**, **3-year**, and **5-year** lookback periods using the NAV at the end of the available history and the NAV at the start of each period (or the earliest available date if history is shorter than the target period).

In [ ]:
def compute_cagr(nav_series, dates, years):
    """Compute CAGR over a trailing period of `years` calendar years."""
    end_date = dates.iloc[-1]
    start_target = end_date - pd.DateOffset(years=years)
    mask = dates >= start_target
    if mask.sum() < 2:
        return np.nan
    start_nav = nav_series[mask].iloc[0]
    end_nav = nav_series.iloc[-1]
    actual_years = (end_date - dates[mask].iloc[0]).days / 365.25
    if start_nav <= 0 or actual_years <= 0:
        return np.nan
    return (end_nav / start_nav) ** (1 / actual_years) - 1

cagr_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    grp = grp.sort_values("date")
    cagr_rows.append({
        "amfi_code": code,
        "cagr_1yr": compute_cagr(grp["nav"], grp["date"], 1),
        "cagr_3yr": compute_cagr(grp["nav"], grp["date"], 3),
        "cagr_5yr": compute_cagr(grp["nav"], grp["date"], 5),
    })

cagr_df = pd.DataFrame(cagr_rows)
cagr_df = cagr_df.merge(fund_df[["amfi_code", "scheme_name", "fund_house"]], on="amfi_code")
cagr_df["cagr_1yr_pct"] = (cagr_df["cagr_1yr"] * 100).round(2)
cagr_df["cagr_3yr_pct"] = (cagr_df["cagr_3yr"] * 100).round(2)
cagr_df["cagr_5yr_pct"] = (cagr_df["cagr_5yr"] * 100).round(2)

cagr_table = cagr_df[["amfi_code", "scheme_name", "fund_house",
                       "cagr_1yr_pct", "cagr_3yr_pct", "cagr_5yr_pct"]].sort_values(
    "cagr_3yr_pct", ascending=False
)
print("CAGR Comparison Table (sorted by 3-year CAGR):")
display(cagr_table)

## 3. Sharpe Ratio

The **Sharpe Ratio** measures risk-adjusted return — how much excess return you earn per unit of total risk (volatility):

$$\text{Sharpe} = \frac{\text{annualized return} - 0.065}{\text{annualized volatility}}$$

- **Annualized return** = mean daily return × 252
- **Annualized volatility** = standard deviation of daily returns × √252
- **Risk-free rate** = 6.5% (0.065)

A higher Sharpe ratio means better return relative to risk. Funds are ranked from highest to lowest Sharpe.

In [ ]:
sharpe_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    rets = grp["daily_return"].dropna()
    ann_return = rets.mean() * TRADING_DAYS
    ann_vol = rets.std() * np.sqrt(TRADING_DAYS)
    sharpe = (ann_return - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else np.nan
    sharpe_rows.append({"amfi_code": code, "annualized_return": ann_return,
                        "annualized_volatility": ann_vol, "sharpe_ratio": sharpe})

sharpe_df = pd.DataFrame(sharpe_rows)
sharpe_df = sharpe_df.merge(fund_df[["amfi_code", "scheme_name"]], on="amfi_code")
sharpe_df["sharpe_rank"] = sharpe_df["sharpe_ratio"].rank(ascending=False, method="min").astype(int)
sharpe_df = sharpe_df.sort_values("sharpe_ratio", ascending=False)

print("Sharpe Ratio Ranking (top 10):")
display(sharpe_df[["sharpe_rank", "scheme_name", "annualized_return",
                     "annualized_volatility", "sharpe_ratio"]].head(10))

## 4. Sortino Ratio

The **Sortino Ratio** is similar to Sharpe but only penalizes **downside** volatility (negative returns), not upside swings:

$$\text{Sortino} = \frac{\text{annualized return} - 0.065}{\text{downside std} \times \sqrt{252}}$$

**Downside standard deviation** is calculated using only daily returns that are below zero. This gives a fairer picture for investors who care more about losses than gains.

In [ ]:
sortino_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    rets = grp["daily_return"].dropna()
    ann_return = rets.mean() * TRADING_DAYS
    downside = rets[rets < 0]
    downside_std = downside.std()
    sortino = ((ann_return - RISK_FREE_RATE) / (downside_std * np.sqrt(TRADING_DAYS))
               if downside_std > 0 else np.nan)
    sortino_rows.append({"amfi_code": code, "sortino_ratio": sortino})

sortino_df = pd.DataFrame(sortino_rows)
sortino_df = sortino_df.merge(fund_df[["amfi_code", "scheme_name"]], on="amfi_code")
sortino_df = sortino_df.sort_values("sortino_ratio", ascending=False)

print("Sortino Ratio (top 10):")
display(sortino_df.head(10))

## 5. Alpha & Beta vs Nifty 100

**Beta** measures how sensitive a fund is to market movements. **Alpha** measures excess return beyond what beta alone would predict.

We run a linear regression of fund daily returns against **Nifty 100** daily returns using `scipy.stats.linregress`:

- **Beta** = slope of the regression
- **Alpha** = intercept × 252 (annualized)

A beta of 1.0 means the fund moves with the market. Alpha > 0 means the fund outperformed its expected return given its market exposure.

In [ ]:
nifty100 = bench_df[bench_df["index_name"] == "NIFTY100"].copy()
nifty100 = nifty100.sort_values("date")
nifty100["daily_return"] = (nifty100["close_value"] / nifty100["close_value"].shift(1)) - 1
nifty100 = nifty100[["date", "daily_return"]].rename(columns={"daily_return": "bench_return"})

alpha_beta_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    fund_rets = grp[["date", "daily_return"]].dropna()
    merged = fund_rets.merge(nifty100, on="date", how="inner").dropna()
    if len(merged) < 30:
        alpha_beta_rows.append({"amfi_code": code, "alpha": np.nan, "beta": np.nan,
                                "r_squared": np.nan, "observations": len(merged)})
        continue
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        merged["bench_return"], merged["daily_return"]
    )
    alpha_beta_rows.append({
        "amfi_code": code,
        "alpha": intercept * TRADING_DAYS,
        "beta": slope,
        "r_squared": r_value ** 2,
        "observations": len(merged),
    })

alpha_beta_df = pd.DataFrame(alpha_beta_rows)
alpha_beta_df = alpha_beta_df.merge(
    fund_df[["amfi_code", "scheme_name", "fund_house", "category"]], on="amfi_code"
)
alpha_beta_df["alpha_pct"] = (alpha_beta_df["alpha"] * 100).round(2)
alpha_beta_df["beta"] = alpha_beta_df["beta"].round(3)
alpha_beta_df = alpha_beta_df.sort_values("alpha", ascending=False)

alpha_beta_df.to_csv(DATA_DIR / "alpha_beta.csv", index=False)
print(f"Saved: {DATA_DIR / 'alpha_beta.csv'}")
display(alpha_beta_df[["scheme_name", "alpha_pct", "beta", "r_squared"]].head(10))

## 6. Maximum Drawdown

**Maximum drawdown** is the largest peak-to-trough decline in NAV — it shows the worst loss an investor would have experienced:

1. Compute a **running maximum** NAV (the highest NAV seen so far)
2. Drawdown = (current NAV − running max) / running max
3. **Max drawdown** = the most negative drawdown value

We also record the dates when the worst drawdown occurred (peak date and trough date).

In [ ]:
mdd_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    grp = grp.sort_values("date").copy()
    running_max = grp["nav"].cummax()
    drawdown = (grp["nav"] - running_max) / running_max
    trough_idx = drawdown.idxmin()
    trough_date = grp.loc[trough_idx, "date"]
    trough_nav = grp.loc[trough_idx, "nav"]
    peak_mask = grp["date"] <= trough_date
    peak_idx = grp.loc[peak_mask, "nav"].idxmax()
    peak_date = grp.loc[peak_idx, "date"]
    mdd_rows.append({
        "amfi_code": code,
        "max_drawdown_pct": (drawdown.min() * 100).round(2),
        "peak_date": peak_date.date(),
        "trough_date": trough_date.date(),
    })

mdd_df = pd.DataFrame(mdd_rows)
mdd_df = mdd_df.merge(fund_df[["amfi_code", "scheme_name"]], on="amfi_code")
mdd_df = mdd_df.sort_values("max_drawdown_pct")

print("Maximum Drawdown (worst 10):")
display(mdd_df.head(10))

## 7. Fund Scorecard (0–100)

The **Fund Scorecard** combines five ranked metrics into a single score from 0 to 100:

| Component | Weight | Direction |
|-----------|--------|-----------|
| 3-year CAGR rank | 30% | Higher is better |
| Sharpe rank | 25% | Higher is better |
| Alpha rank | 20% | Higher is better |
| Expense Ratio rank | 15% | Lower is better (inverse) |
| Max Drawdown rank | 10% | Smaller loss is better (inverse) |

Each metric is converted to a percentile rank (0–100), then weighted and summed.

In [ ]:
scorecard = fund_df[["amfi_code", "scheme_name", "fund_house", "expense_ratio_pct"]].copy()
scorecard = scorecard.merge(cagr_df[["amfi_code", "cagr_3yr"]], on="amfi_code")
scorecard = scorecard.merge(sharpe_df[["amfi_code", "sharpe_ratio"]], on="amfi_code")
scorecard = scorecard.merge(alpha_beta_df[["amfi_code", "alpha"]], on="amfi_code")
scorecard = scorecard.merge(mdd_df[["amfi_code", "max_drawdown_pct"]], on="amfi_code")

scorecard["cagr_rank"] = scorecard["cagr_3yr"].rank(pct=True) * 100
scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(pct=True) * 100
scorecard["alpha_rank"] = scorecard["alpha"].rank(pct=True) * 100
n_funds = len(scorecard)
expense_r = scorecard["expense_ratio_pct"].rank(ascending=True, method="min")
scorecard["expense_rank"] = (n_funds - expense_r + 1) / n_funds * 100
scorecard["mdd_rank"] = scorecard["max_drawdown_pct"].rank(pct=True) * 100

scorecard["fund_score"] = (
    0.30 * scorecard["cagr_rank"]
    + 0.25 * scorecard["sharpe_rank"]
    + 0.20 * scorecard["alpha_rank"]
    + 0.15 * scorecard["expense_rank"]
    + 0.10 * scorecard["mdd_rank"]
).round(2)

scorecard["overall_rank"] = scorecard["fund_score"].rank(ascending=False, method="min").astype(int)
scorecard = scorecard.sort_values("fund_score", ascending=False)

scorecard_out = scorecard[[
    "overall_rank", "amfi_code", "scheme_name", "fund_house", "fund_score",
    "cagr_rank", "sharpe_rank", "alpha_rank", "expense_rank", "mdd_rank",
    "cagr_3yr", "sharpe_ratio", "alpha", "expense_ratio_pct", "max_drawdown_pct",
]].copy()
scorecard_out["cagr_3yr_pct"] = (scorecard_out["cagr_3yr"] * 100).round(2)
scorecard_out["alpha_pct"] = (scorecard_out["alpha"] * 100).round(2)

scorecard_out.to_csv(DATA_DIR / "fund_scorecard.csv", index=False)
print(f"Saved: {DATA_DIR / 'fund_scorecard.csv'}")
display(scorecard_out[["overall_rank", "scheme_name", "fund_score",
                          "cagr_3yr_pct", "sharpe_ratio", "alpha_pct"]].head(10))

## 8. Benchmark Comparison Chart

We plot the **top 5 funds** from the scorecard against **Nifty 50** and **Nifty 100** over the last 3 years.

All series are normalized to 100 at the start date for fair comparison.

**Tracking error** measures how closely a fund follows a benchmark:

$$\text{tracking\_error} = \text{std}(\text{fund return} - \text{benchmark return}) \times \sqrt{252}$$

Lower tracking error means the fund's returns are closer to the benchmark.

In [ ]:
top5_codes = scorecard_out.head(5)["amfi_code"].tolist()
end_date = nav_df["date"].max()
start_date = end_date - pd.DateOffset(years=3)

def normalized_growth(series, dates):
    """Normalize a price series to 100 at the first value."""
    return (series / series.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 7))

tracking_errors = []

for idx_name, color, linestyle in [("NIFTY50", "black", "--"), ("NIFTY100", "gray", ":")]:
    bench = bench_df[(bench_df["index_name"] == idx_name) & (bench_df["date"] >= start_date)].copy()
    bench = bench.sort_values("date")
    bench["indexed"] = normalized_growth(bench["close_value"], bench["date"])
    ax.plot(bench["date"], bench["indexed"], label=idx_name.replace("NIFTY", "Nifty "),
            color=color, linestyle=linestyle, linewidth=2)

nifty100_3yr = bench_df[(bench_df["index_name"] == "NIFTY100") & (bench_df["date"] >= start_date)].copy()
nifty100_3yr = nifty100_3yr.sort_values("date")
nifty100_3yr["daily_return"] = (nifty100_3yr["close_value"] / nifty100_3yr["close_value"].shift(1)) - 1

colors = plt.cm.tab10.colors
for i, code in enumerate(top5_codes):
    fund_nav = nav_df[(nav_df["amfi_code"] == code) & (nav_df["date"] >= start_date)].copy()
    fund_nav = fund_nav.sort_values("date")
    name = fund_df.loc[fund_df["amfi_code"] == code, "scheme_name"].values[0]
    short_name = name.split(" - ")[0][:30]
    fund_nav["indexed"] = normalized_growth(fund_nav["nav"], fund_nav["date"])
    ax.plot(fund_nav["date"], fund_nav["indexed"], label=short_name, color=colors[i], linewidth=1.5)

    fund_rets = fund_nav[["date", "daily_return"]].dropna()
    merged_te = fund_rets.merge(
        nifty100_3yr[["date", "daily_return"]], on="date", how="inner",
        suffixes=("_fund", "_bench")
    ).dropna()
    if len(merged_te) > 1:
        te = merged_te["daily_return_fund"].sub(merged_te["daily_return_bench"]).std() * np.sqrt(TRADING_DAYS)
        tracking_errors.append({"scheme_name": short_name, "vs_nifty100_te": round(te * 100, 2)})

ax.set_title("Top 5 Funds vs Nifty 50 & Nifty 100 (Last 3 Years, Indexed to 100)")
ax.set_xlabel("Date")
ax.set_ylabel("Indexed Value (Start = 100)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {CHART_DIR / 'benchmark_comparison.png'}")
print("\nTracking Error vs Nifty 100 (top 5 funds):")
display(pd.DataFrame(tracking_errors))

## Summary

This notebook computed performance analytics for all 40 mutual fund schemes:

- **Daily returns** validated with summary statistics and histogram
- **CAGR** for 1-year, 3-year, and 5-year periods
- **Sharpe** and **Sortino** ratios for risk-adjusted ranking
- **Alpha & Beta** vs Nifty 100 saved to `data/processed/alpha_beta.csv`
- **Maximum drawdown** with peak and trough dates
- **Fund scorecard** saved to `data/processed/fund_scorecard.csv`
- **Benchmark comparison chart** saved to `reports/charts/benchmark_comparison.png`